# ST-TR - Spatial-Temporal Transformer, single stream

Based on Plizzari, Cannici & Matteucci (2021). Their official implementation trains Spatial Self-Attention (SSA) and Temporal Self-Attention (TSA) as two separate streams and ensembles the scores - this instead follows their own documented single-stream variant (combining both SSA and TSA in one model, their Sec. V(F)), so this is a genuine single model, not an ensemble.

**VideoBadminton (Li et al.) does not benchmark ST-TR** - their comparison covers R(2+1)D, SlowFast, TimeSformer, Swin, MViT-V2, ST-GCN, and PoseC3D. TimeSformer/Swin/MViT-V2 are RGB-frame transformers, a different modality from skeleton-graph attention. This model has no external benchmark number to chase - compare it against this project's own LSTM and ST-GCN results instead.

Backbone shape mirrors the project's verified ST-GCN exactly: 9 stages, 64/64/64 -> 128/128/128 -> 256/256/256, stride-2 downsampling at stages 4 and 7 - confirmed via a real shape trace before this notebook was built, not assumed. Same proven recipe as ST-GCN's best result otherwise: `joint_bone` (5-channel) input, `weight_decay=1e-3`, StepLR, advanced augmentation, sampler built fresh per call, seeded workers.

### determinism - before torch touches CUDA

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, top_k_accuracy_score,
    confusion_matrix, precision_recall_fscore_support,
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sys.path.insert(0, "/workspace/badminton-honours-project")
import dataset as ds

from clearml import Task

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda:0


### DataLoader worker-seeding fix

In [2]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


### paths

In [3]:
OUT_ROOT = "/workspace/outputs/sttr"
CKPT_ROOT = "/workspace/checkpoints/sttr"
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)


### data - advanced augmentation, joint_bone, 30-frame

In [4]:
manifest = ds.load_filtered_manifest(
    "/workspace/data/split_manifest.csv",
    "/workspace/data/extraction_log.csv",
)

class_names = sorted(manifest["class_name"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
num_classes = len(class_names)

train_manifest = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_manifest = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_manifest = manifest[manifest["split"] == "test"].reset_index(drop=True)

KEYPOINTS_ROOT = "/workspace/data/keypoints_smoothed"

train_ds = ds.BadmintonPoseDataset(train_manifest, KEYPOINTS_ROOT, class_to_id, augmentation_policy="advanced")
val_ds = ds.BadmintonPoseDataset(val_manifest, KEYPOINTS_ROOT, class_to_id)
test_ds = ds.BadmintonPoseDataset(test_manifest, KEYPOINTS_ROOT, class_to_id)

val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"{num_classes} classes, train {len(train_manifest)}, val {len(val_ds)}, test {len(test_ds)}")


dropped 4 manually-excluded clips from the manifest (7814 remain)
18 classes, train 6251, val 781, test 782


### the ST-TR model

SSA replaces the spatial graph convolution (joint-to-joint attention within a frame); TSA replaces the temporal convolution (frame-to-frame attention across time), applied sequentially within each block. Verified with a real forward+backward pass and a full shape trace before this notebook was written - not shipped untested.

In [5]:
class SpatialSelfAttention(nn.Module):
    """Joint-to-joint self-attention within each frame, independently per frame."""

    def __init__(self, channels, num_heads=8, dropout=0.0):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=channels, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        N, C, T, V = x.shape
        x_flat = x.permute(0, 2, 3, 1).reshape(N * T, V, C)
        attn_out, _ = self.mha(x_flat, x_flat, x_flat)
        out = self.norm(x_flat + attn_out)
        out = out.reshape(N, T, V, C).permute(0, 3, 1, 2)
        return out


class TemporalSelfAttention(nn.Module):
    """Frame-to-frame self-attention across time, independently per joint."""

    def __init__(self, channels, num_heads=8, dropout=0.0):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=channels, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        N, C, T, V = x.shape
        x_flat = x.permute(0, 3, 2, 1).reshape(N * V, T, C)
        attn_out, _ = self.mha(x_flat, x_flat, x_flat)
        out = self.norm(x_flat + attn_out)
        out = out.reshape(N, V, T, C).permute(0, 3, 2, 1)
        return out


class STTRBlock(nn.Module):
    """One ST-TR block: SSA then TSA, sequentially."""

    def __init__(self, in_channels, out_channels, num_heads=8, stride=1, dropout=0.0):
        super().__init__()
        self.in_proj = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else nn.Identity()
        )
        self.ssa = SpatialSelfAttention(out_channels, num_heads=num_heads, dropout=dropout)
        self.tsa = TemporalSelfAttention(out_channels, num_heads=num_heads, dropout=dropout)

        self.pool = nn.AvgPool2d(kernel_size=(stride, 1), stride=(stride, 1)) if stride > 1 else nn.Identity()

        if in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1),
                nn.AvgPool2d(kernel_size=(stride, 1), stride=(stride, 1)) if stride > 1 else nn.Identity(),
            )

        self.relu = nn.ReLU()

    def forward(self, x):
        res = self.residual(x)
        out = self.in_proj(x)
        out = self.ssa(out)
        out = self.tsa(out)
        out = self.pool(out)
        return self.relu(out + res)


class STTR(nn.Module):
    def __init__(self, in_channels=5, num_classes=18, base_channels=64,
                 num_stages=9, inflate_stages=(4, 7), down_stages=(4, 7),
                 num_heads=8, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Conv2d(in_channels, base_channels, kernel_size=1)

        blocks = []
        channels = base_channels
        for stage in range(1, num_stages + 1):
            out_channels = channels * 2 if stage in inflate_stages else channels
            stride = 2 if stage in down_stages else 1
            blocks.append(STTRBlock(channels, out_channels, num_heads=num_heads,
                                     stride=stride, dropout=dropout))
            channels = out_channels

        self.blocks = nn.ModuleList(blocks)
        self.final_channels = channels

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(channels, num_classes)

    def forward(self, x):
        out = self.input_proj(x)
        for block in self.blocks:
            out = block(out)
        out = self.pool(out).flatten(1)
        return self.fc(out)


### batch -> ST-TR input

Different convention from the ST-GCN notebooks - plain (N, C, T, V), not the 6-D RecognizerGCN shape, since this is a plain nn.Module, not routed through mmaction2's registry.

In [6]:
def batch_to_sttr_input(batch):
    x = torch.cat([batch["keypoints"].float(), batch["bone"].float()], dim=-1)  # (N, T, V, C=5)
    return x.permute(0, 3, 1, 2)  # (N, C, T, V)


### SMOKE TEST - run alone first

Already verified standalone with real torch execution before this notebook was written (forward+backward pass, full shape trace through all 9 blocks) - this re-confirms it works with the actual project data pipeline specifically, not synthetic tensors.

In [7]:
smoke_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
smoke_batch = next(iter(smoke_loader))
smoke_input = batch_to_sttr_input(smoke_batch).to(device)
print("input shape:", smoke_input.shape)
assert smoke_input.shape[1] == 5, f"expected 5 channels, got {smoke_input.shape[1]}"

smoke_model = STTR(in_channels=5, num_classes=num_classes).to(device)
n_params = sum(p.numel() for p in smoke_model.parameters())
print(f"model built, {n_params:,} parameters")

smoke_model.eval()
with torch.no_grad():
    smoke_out = smoke_model(smoke_input)
print("output shape:", smoke_out.shape)
assert smoke_out.shape == (4, num_classes), f"expected (4, {num_classes}), got {smoke_out.shape}"

smoke_model.train()
smoke_out = smoke_model(smoke_input)
smoke_out.sum().backward()
n_with_grad = sum(1 for p in smoke_model.parameters() if p.grad is not None)
n_total = sum(1 for p in smoke_model.parameters())
print(f"backward pass: {n_with_grad}/{n_total} parameters received gradients")
assert n_with_grad == n_total, "some parameters did not receive gradients"

del smoke_model
print()
print("smoke test passed")


input shape: torch.Size([4, 5, 30, 17])
model built, 2,168,210 parameters
output shape: torch.Size([4, 18])
backward pass: 120/120 parameters received gradients

smoke test passed


### metrics

In [8]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for batch in loader:
        x = batch_to_sttr_input(batch).to(device)
        y = batch["label"].to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)

        all_labels.append(y.cpu().numpy())
        all_preds.append(probs.argmax(dim=1).cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    labels = np.concatenate(all_labels)
    preds = np.concatenate(all_preds)
    probs = np.concatenate(all_probs)

    return {
        "top1": accuracy_score(labels, preds),
        "top5": top_k_accuracy_score(labels, probs, k=5, labels=list(range(num_classes))),
        "mca": balanced_accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "labels": labels, "preds": preds,
    }


### training - same proven recipe as ST-GCN's best single-model result

In [9]:
def train_sttr(seed=42, num_epochs=60, patience=20, batch_size=16,
                lr=1e-4, weight_decay=1e-3, grad_clip_norm=1.0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    balanced_sampler = ds.build_class_balanced_sampler(train_manifest, target_per_class=500, seed=seed)
    model = STTR(in_channels=5, num_classes=num_classes).to(device)

    loader_generator = torch.Generator()
    loader_generator.manual_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, sampler=balanced_sampler,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=seed_worker, generator=loader_generator,
    )

    task = Task.init(project_name="BadmintonPoseAnalysis", task_name="sttr_joint_bone", reuse_last_task_id=False)
    logger = task.get_logger()
    task.connect({
        "model": "ST-TR", "stream": "joint_bone", "seed": seed,
        "optimizer": "AdamW", "schedule": "cosine", "grad_clip_norm": grad_clip_norm,
        "lr": lr, "weight_decay": weight_decay, "num_epochs": num_epochs,
        "batch_size": batch_size, "augmentation_policy": "advanced",
    })

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=0)

    best_val_top1 = 0.0
    best_epoch = -1
    epochs_without_improvement = 0
    best_state = None

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_labels, epoch_preds = [], []

        for batch in train_loader:
            x = batch_to_sttr_input(batch).to(device)
            y = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
            optimizer.step()

            epoch_loss += loss.item() * x.size(0)
            epoch_labels.append(y.cpu().numpy())
            epoch_preds.append(logits.argmax(dim=1).cpu().numpy())

        scheduler.step()

        train_loss = epoch_loss / len(train_loader.dataset)
        train_acc = accuracy_score(np.concatenate(epoch_labels), np.concatenate(epoch_preds))
        val_metrics = evaluate(model, val_loader)

        logger.report_scalar("loss", "train", train_loss, iteration=epoch)
        logger.report_scalar("accuracy", "train", train_acc, iteration=epoch)
        logger.report_scalar("accuracy", "val_top1", val_metrics["top1"], iteration=epoch)
        logger.report_scalar("accuracy", "val_mca", val_metrics["mca"], iteration=epoch)

        if val_metrics["top1"] > best_val_top1:
            best_val_top1 = val_metrics["top1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1

        print(f"epoch {epoch:3d}  train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
              f"val_top1={val_metrics['top1']:.4f}  val_mca={val_metrics['mca']:.4f}")

        if epochs_without_improvement >= patience:
            print(f"early stopping at epoch {epoch}, best val_top1={best_val_top1:.4f} (epoch {best_epoch})")
            break

    model.load_state_dict(best_state)

    train_final = evaluate(model, DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True))
    test_final = evaluate(model, test_loader)

    metrics = {
        "model": "ST-TR",
        "best_epoch": best_epoch,
        "train_acc": train_final["top1"],
        "test_top1": test_final["top1"],
        "test_top5": test_final["top5"],
        "test_mca": test_final["mca"],
        "test_macro_f1": test_final["macro_f1"],
        "train_test_gap": train_final["top1"] - test_final["top1"],
    }

    for metric_name in ["train_acc", "test_top1", "test_top5", "test_mca", "test_macro_f1", "train_test_gap", "best_epoch"]:
        logger.report_single_value(name=metric_name, value=metrics[metric_name])

    cm = confusion_matrix(test_final["labels"], test_final["preds"], labels=list(range(num_classes)))
    logger.report_confusion_matrix(
        title="Test Confusion Matrix", series="ST-TR",
        matrix=cm, xaxis="Predicted", yaxis="Actual",
        xlabels=class_names, ylabels=class_names,
    )

    precision, recall, f1, support = precision_recall_fscore_support(
        test_final["labels"], test_final["preds"], labels=list(range(num_classes)), zero_division=0,
    )
    per_class_df = pd.DataFrame({
        "class": class_names, "precision": precision, "recall": recall,
        "f1": f1, "support": support,
    })
    logger.report_table(title="Per-Class Test Metrics", series="ST-TR", table_plot=per_class_df)

    ckpt_path = os.path.join(CKPT_ROOT, "sttr_joint_bone_best.pt")
    torch.save(model.state_dict(), ckpt_path)
    task.upload_artifact("checkpoint", ckpt_path)
    per_class_df.to_csv(os.path.join(OUT_ROOT, "per_class_sttr.csv"), index=False)
    task.close()

    return metrics


### run it

In [10]:
metrics = train_sttr()
print()
print(f"FINAL: test_top1={metrics['test_top1']:.4f}  gap={metrics['train_test_gap']:.4f}  "
      f"test_mca={metrics['test_mca']:.4f}  test_macro_f1={metrics['test_macro_f1']:.4f}")


ClearML Task: created new task id=7219b1ca77524789a3c7e4e85773a64f
2026-09-23 08:23:04,992 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/7219b1ca77524789a3c7e4e85773a64f/output/log
epoch   0  train_loss=5.7824  train_acc=0.1089  val_top1=0.0948  val_mca=0.1162
epoch   1  train_loss=4.4087  train_acc=0.1596  val_top1=0.1165  val_mca=0.1457
epoch   2  train_loss=3.9961  train_acc=0.2353  val_top1=0.2356  val_mca=0.2323
epoch   3  train_loss=3.5697  train_acc=0.3077  val_top1=0.3060  val_mca=0.2881
epoch   4  train_loss=3.2710  train_acc=0.3586  val_top1=0.4110  val_mca=0.3738
epoch   5  train_loss=3.0565  train_acc=0.3955  val_top1=0.4469  val_mca=0.3837
epoch   6  train_loss=2.8747  train_acc=0.4380  val_top1=0.5032  val_mca=0.4322
epoch   7  train_loss=2.7084  train_acc=0.4742  val_top1=0.5416  val_mca=0.4520
epoch   8  train_loss=2.5235  train_acc=0.5000  

█████████████████████████████████ 100% | 8.31/8.31 MB [00:02<00:00,  3.89MB/s]: 



FINAL: test_top1=0.6343  gap=0.2934  test_mca=0.5209  test_macro_f1=0.5342


### comparison against this project's own models

In [11]:
PRIOR_RESULTS = pd.DataFrame([
    {"model": "ST-GCN (joint_bone, 3-seed mean)", "test_top1": 0.7089, "test_top5": 0.9442,
     "test_mca": 0.6222, "test_macro_f1": 0.6340},
    {"model": metrics["model"], "test_top1": metrics["test_top1"], "test_top5": metrics["test_top5"],
     "test_mca": metrics["test_mca"], "test_macro_f1": metrics["test_macro_f1"]},
]).set_index("model")

PRIOR_RESULTS.to_csv(os.path.join(OUT_ROOT, "sttr_vs_stgcn.csv"))
PRIOR_RESULTS


,test_top1,test_top5,test_mca,test_macro_f1
model,,,,
"ST-GCN (joint_bone, 3-seed mean)",0.708900,0.944200,0.62220,0.634000
ST-TR,0.634271,0.902813,0.52095,0.534214


---

This is a single-seed result - if it looks competitive with or better than ST-GCN, it's worth a multi-seed confirmation before treating the comparison as trustworthy, same pattern as every other result today. No external benchmark exists for this model, so the comparison that matters is against this project's own LSTM and ST-GCN numbers.